In [ ]:
"""
Process lymphoma CosMx data into gridded or single-cell formats.

Core purpose: Process an image and h5ad file into spatially resolved outputs.
"""

import logging
import re
import shutil
from pathlib import Path

import anndata
import numpy as np
import pandas as pd
from PIL import Image

from coordinate_utils import convert_mm_to_pixel_coordinates
from image_utils import (
    OpenSlideWrapper,
    handle_magnification,
    crop_tile,
    prepare_visualization_image,
)
from grid_utils import (
    create_grid_coordinates,
    filter_background_tiles,
    aggregate_expression_data,
)
from visualization import generate_qc_tile_plot, save_example_patches

logging.basicConfig(level=logging.INFO)


def sanitize_identifier(value: str) -> str:
    """Normalize identifiers for file-safe usage."""
    sanitized = re.sub(r"[^A-Za-z0-9_-]", "_", str(value))
    return sanitized or "cell"


# Get parameters from snakemake
dataset = snakemake.wildcards.dataset
IS_SINGLECELL = dataset.endswith("_singlecell")
SPOT_DIAMETER_PIXELS = snakemake.params.spot_diameter_pixels
WHITE_CUTOFF = snakemake.params.white_cutoff
X_OFFSET = snakemake.params.x_offset
Y_OFFSET = snakemake.params.y_offset
SCALE_FACTOR = snakemake.params.scale_factor
FILTER_GOOD_QUALITY = snakemake.params.filter_good_quality
PIXEL_SIZE_UM = snakemake.params.pixel_size_um
SMALL_PATCH_DIAMETER_UM = snakemake.params.small_patch_diameter_um
MEDIUM_PATCH_PIXELS = snakemake.params.medium_patch_pixels
LARGE_PATCH_PIXELS = snakemake.params.large_patch_pixels
MAX_PATCHES_PER_SCALE = snakemake.params.max_patches_per_scale
if MAX_PATCHES_PER_SCALE in (None, "None"):
    MAX_PATCHES_PER_SCALE = None
else:
    MAX_PATCHES_PER_SCALE = int(MAX_PATCHES_PER_SCALE)

logging.info(
    f"Processing {dataset} | spot_diameter_pixels={SPOT_DIAMETER_PIXELS}, "
    f"white_cutoff={WHITE_CUTOFF}, x_offset={X_OFFSET}, y_offset={Y_OFFSET}, scale_factor={SCALE_FACTOR}, "
    f"filter_good_quality={FILTER_GOOD_QUALITY}, pixel_size={PIXEL_SIZE_UM} μm/pixel"
)

# Ensure patch output directory exists (used for single-cell variant)
patch_output_root = Path(snakemake.output.patch_dir)
patch_output_root.mkdir(parents=True, exist_ok=True)

# Load data
logging.info(f"Loading AnnData from {snakemake.input.read_count_table}")
data_hr = anndata.read_h5ad(snakemake.input.read_count_table)
adata_hr = data_hr

# Map cell barcodes to core IDs
core_assignment_path = getattr(snakemake.input, "cell_barcode_core_assignment", None)
if core_assignment_path:
    logging.info(
        f"Loading cell barcode to core mapping from {core_assignment_path}"
    )
    core_df = pd.read_csv(core_assignment_path)
    required_cols = {"cell_barcode", "core_id"}
    missing_cols = required_cols - set(core_df.columns)
    if missing_cols:
        raise ValueError(
            f"Missing columns {missing_cols} in {core_assignment_path}"
        )
    barcode_to_core = dict(
        zip(core_df["cell_barcode"].astype(str), core_df["core_id"].astype(str))
    )
    adata_hr.obs["core_id"] = adata_hr.obs_names.astype(str).map(barcode_to_core)
    missing_core_ids = adata_hr.obs["core_id"].isna().sum()
    if missing_core_ids > 0:
        logging.warning(
            f"{missing_core_ids} cells did not have a matching core assignment"
        )
else:
    logging.warning(
        "No cell barcode to core mapping provided; core_id will remain empty"
    )

# Filter for good quality cells if requested
if FILTER_GOOD_QUALITY and "good_quality" in adata_hr.obs.columns:
    n_cells_before = adata_hr.n_obs
    adata_hr = adata_hr[adata_hr.obs["good_quality"]].copy()
    n_cells_after = adata_hr.n_obs
    logging.info(
        f"Filtered for good_quality cells: {n_cells_before} -> {n_cells_after} cells"
    )
elif FILTER_GOOD_QUALITY:
    logging.warning(
        "good_quality column not found in adata.obs, skipping quality filtering"
    )

# Record pixel size metadata before any magnification adjustments
adata_hr.uns["pixel_size"] = PIXEL_SIZE_UM

logging.info(f"Loading image from {snakemake.input.image}")
slide_wrapper = OpenSlideWrapper(snakemake.input.image, dataset)

# Handle magnification scaling
original_width, original_height = slide_wrapper.slide.level_dimensions[
    0
]  # Level 0 dimensions
slide_wrapper = handle_magnification(adata_hr, slide_wrapper, dataset)

# Convert coordinates from mm to pixels
width, height = slide_wrapper.size
image_was_scaled = (width != original_width) or (height != original_height)

if image_was_scaled:
    logging.info("Image was scaled - adjusting coordinate transformation")
    spatial_coords = convert_mm_to_pixel_coordinates(
        adata_hr, original_width, original_height, X_OFFSET, Y_OFFSET, SCALE_FACTOR
    )
    # Scale coordinates to match the scaled image
    scale_factor_x = width / original_width
    scale_factor_y = height / original_height
    spatial_coords[:, 0] *= scale_factor_x
    spatial_coords[:, 1] *= scale_factor_y
    logging.info(
        f"Scaled coordinates by factors: x={scale_factor_x:.3f}, y={scale_factor_y:.3f}"
    )
else:
    logging.info("Image was not scaled - using direct coordinate transformation")
    spatial_coords = convert_mm_to_pixel_coordinates(
        adata_hr, width, height, X_OFFSET, Y_OFFSET, SCALE_FACTOR
    )

adata_hr.obsm["spatial"] = spatial_coords

# Prepare visualization image
img_np, scale_factor_img = prepare_visualization_image(slide_wrapper)

coord_df = None
filtered_coords = None
output_adata = None

if IS_SINGLECELL:
    logging.info("Entering single-cell processing branch (40x resolution)")

    # Determine patch sizes in pixels
    pixel_size_um = adata_hr.uns["pixel_size"]
    patch_sizes = {
        "small": int(round(SMALL_PATCH_DIAMETER_UM / pixel_size_um)),
        "medium": int(MEDIUM_PATCH_PIXELS),
        "large": int(LARGE_PATCH_PIXELS),
    }

    for scale, size in patch_sizes.items():
        if size < 1:
            patch_sizes[scale] = 1
        if patch_sizes[scale] % 2 != 0:
            patch_sizes[scale] += 1  # ensure even size for symmetric cropping
        logging.info(
            f"Patch '{scale}': {patch_sizes[scale]} px (~{patch_sizes[scale] * pixel_size_um:.2f} μm)"
        )

    total_cells = adata_hr.n_obs
    if MAX_PATCHES_PER_SCALE is not None:
        sample_count = min(int(MAX_PATCHES_PER_SCALE), total_cells)
        rng = np.random.default_rng(42)
        selected_indices = np.sort(
            rng.choice(total_cells, size=sample_count, replace=False)
        )
        process_mask = np.zeros(total_cells, dtype=bool)
        process_mask[selected_indices] = True
        logging.info(
            f"Sampling {sample_count} cells (of {total_cells}) for patch extraction"
        )
    else:
        process_mask = np.ones(total_cells, dtype=bool)

    patch_dirs = {}
    for scale in patch_sizes:
        patch_dir = patch_output_root / scale
        patch_dir.mkdir(parents=True, exist_ok=True)
        patch_dirs[scale] = patch_dir

    patch_id_arrays = {
        scale: np.full(total_cells, None, dtype=object) for scale in patch_sizes
    }
    patch_path_arrays = {
        scale: np.full(total_cells, None, dtype=object) for scale in patch_sizes
    }
    saved_patch_paths = {scale: [] for scale in patch_sizes}

    for idx, (cell_id, (x_center, y_center)) in enumerate(
        zip(adata_hr.obs_names, spatial_coords)
    ):
        if not process_mask[idx]:
            continue

        base_id = f"{sanitize_identifier(cell_id)}_{idx:06d}"

        for scale, size in patch_sizes.items():
            patch_id = f"{base_id}_{scale}"
            patch_path = patch_dirs[scale] / f"{patch_id}.png"

            top_left_x = int(round(x_center - size / 2))
            top_left_y = int(round(y_center - size / 2))
            patch_array = crop_tile(slide_wrapper, top_left_x, top_left_y, size)
            patch_array = patch_array[:, :, :3]

            Image.fromarray(patch_array.astype(np.uint8)).save(patch_path)

            relative_path = Path("patches") / scale / f"{patch_id}.png"
            patch_id_arrays[scale][idx] = patch_id
            patch_path_arrays[scale][idx] = str(relative_path)
            saved_patch_paths[scale].append(str(relative_path))

        if (idx + 1) % 5000 == 0:
            logging.info(f"Processed {idx + 1} / {total_cells} cells (sampled)")

    output_adata = adata_hr.copy()
    for scale in patch_sizes:
        output_adata.obs[f"patch_id_{scale}"] = patch_id_arrays[scale]
        output_adata.obs[f"patch_path_{scale}"] = patch_path_arrays[scale]

    output_adata.uns.setdefault("metadata", {})
    output_adata.uns["metadata"]["patch_output_root"] = str(patch_output_root)
    output_adata.uns["patch_metadata"] = {
        scale: {
            "size_pixels": size,
            "size_um": size * pixel_size_um,
            "description": f"{scale.capitalize()} patch centered on cell centroid",
        }
        for scale, size in patch_sizes.items()
    }
    output_adata.uns["pixel_size"] = pixel_size_um
    output_adata.layers["counts"] = output_adata.X.copy()

else:
    logging.info("Entering gridded aggregation branch")

    coord_df = create_grid_coordinates(width, height, SPOT_DIAMETER_PIXELS)
    filtered_coords = filter_background_tiles(
        coord_df, slide_wrapper, SPOT_DIAMETER_PIXELS, WHITE_CUTOFF, crop_tile
    )

    aggregated_counts, cell_counts_per_tile, fov_info, core_info = (
        aggregate_expression_data(adata_hr, filtered_coords, SPOT_DIAMETER_PIXELS)
    )

    if aggregated_counts:
        counts_matrix = np.vstack(aggregated_counts)
    else:
        counts_matrix = np.empty((0, adata_hr.n_vars))

    gridded_adata = anndata.AnnData(counts_matrix, var=adata_hr.var)
    gridded_adata.obs.index = filtered_coords.index
    gridded_adata.obs["x_array"] = filtered_coords["x_array"]
    gridded_adata.obs["y_array"] = filtered_coords["y_array"]
    gridded_adata.obs["x_pixel"] = filtered_coords["x_pixel"]
    gridded_adata.obs["y_pixel"] = filtered_coords["y_pixel"]
    gridded_adata.obs["n_cells"] = cell_counts_per_tile
    gridded_adata.obs["fov_info"] = fov_info
    gridded_adata.obs["core_info"] = core_info
    gridded_adata.obsm["spatial"] = filtered_coords[["x_pixel", "y_pixel"]].values

    # Filter for min number of cells per tile
    gridded_adata = gridded_adata[
        gridded_adata.obs["n_cells"] >= snakemake.params["min_num_cells"]
    ].copy()

    gridded_adata.uns["pixel_size"] = adata_hr.uns["pixel_size"]
    output_adata = gridded_adata

library_id = "lymphoma_cosmx_small"
output_adata.uns["spatial"] = {
    library_id: {
        "images": {"hires": img_np},
        "scalefactors": {
            "tissue_hires_scalef": scale_factor_img,
            "spot_diameter_fullres": SPOT_DIAMETER_PIXELS,
        },
        "metadata": {"library_id": library_id},
    }
}

# Add image_path to uns and copy the SVS file
output_adata.uns["image_path"] = str(snakemake.output.image)

logging.info(
    f"Copying SVS file from {snakemake.input.image} to {snakemake.output.image}"
)
shutil.copy2(snakemake.input.image, snakemake.output.image)

# Generate outputs
if not IS_SINGLECELL:
    if hasattr(snakemake.output, "qc_tile_plot"):
        generate_qc_tile_plot(
            coord_df,
            img_np,
            SPOT_DIAMETER_PIXELS,
            scale_factor_img,
            snakemake.output.qc_tile_plot,
        )

    if (
        hasattr(snakemake.output, "report_patches")
        and len(snakemake.output.report_patches) > 0
    ):
        save_example_patches(
            filtered_coords,
            slide_wrapper,
            SPOT_DIAMETER_PIXELS,
            snakemake.output.report_patches,
            crop_tile,
        )
else:
    if hasattr(snakemake.output, "qc_tile_plot"):
        Image.fromarray(img_np.astype(np.uint8)).save(snakemake.output.qc_tile_plot)

    if (
        hasattr(snakemake.output, "report_patches")
        and len(snakemake.output.report_patches) > 0
    ):
        dataset_root = patch_output_root.parent
        medium_paths = saved_patch_paths.get("medium") or []
        fallback_paths = next(
            (paths for paths in saved_patch_paths.values() if paths), []
        )
        selected_paths = (
            medium_paths
            if len(medium_paths) >= len(snakemake.output.report_patches)
            else fallback_paths
        )

        if not selected_paths:
            logging.warning("No patched images available for report_patches output")
        else:
            needed = len(snakemake.output.report_patches)
            replicated = (selected_paths * ((needed // len(selected_paths)) + 1))[
                :needed
            ]
            for dest_path, rel_path in zip(
                snakemake.output.report_patches, replicated
            ):
                shutil.copy2(dataset_root / rel_path, dest_path)

# Close the slide
slide_wrapper.close()

# Save processed data
logging.info(f"Saving output AnnData to {snakemake.output.adata}")
output_adata.write_h5ad(snakemake.output.adata)
logging.info("Processing complete.")

